<a href="https://colab.research.google.com/github/Shraddha6211/Machine-Learning-Extra-Class/blob/main/cbow_skipgrams_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import tensorflow as tf
import numpy as np

In [8]:
# Data
text = "i love deep learning and i love machine learning"
tokens = text.split()

vocab = sorted(set(tokens))
word2idx = {w:i for i, w in enumerate(vocab)}
idx2word = {i:w for w, i in word2idx.items()}

vocab_size = len(vocab)
window_size =2

# CBOW DATA
def make_cbow_data(tokens, window):
  X, y = [],[]

  for i in range(window, len(tokens)-window):
    context =[]

    for j in range(-window, window+1):
      if j !=0:
        context.append(word2idx[tokens[i+j]])

    X.append(context)
    y.append(word2idx[tokens[i]])

  return np.array(X), np.array(y)

X, y = make_cbow_data(tokens, window_size)

#   CBOW Model
embedding_dim =10
class CBOW(tf.keras.Model):
  def __init__(self):
    super().__init__()
    self.embedding = tf.keras.layers.Embedding(
        vocab_size, embedding_dim
    )
    self.output_layer = tf.keras.layers.Dense(vocab_size)

  def call(self, x):
    x = self.embedding(x)
    x = tf.reduce_mean(x, axis=1)
    x = self.output_layer(x)
    return x

model = CBOW()

model.compile(
    optimizer = "adam",
    loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics = ["accuracy"]
)

model.fit(X, y, epochs=200, verbose = 0)

embeddings_cbow = model.embedding.get_weights()[0]

In [9]:
context_words = {"i", "deep", "learning", "and"}

context_ids = np.array([word2idx[w] for w in context_words])

logits = model(context_ids)
pred_id = tf.argmax(logits, axis =1).numpy()[0]

print("Predicted words:")


ValueError: Exception encountered when calling CBOW.call().

[1mInput 0 with name 'None' of layer 'dense_2' is incompatible with the layer: expected min_ndim=2, found ndim=1. Full shape received: (4,)[0m

Arguments received by CBOW.call():
  • x=tf.Tensor(shape=(4,), dtype=int64)

In [10]:
embeddings = model.embedding.get_weights()[0]

print("Vector for 'learning':")
print(embeddings[word2idx["learning"]])

Vector for 'learning':
[ 0.0124043  -0.3622229  -0.09912224  0.1668182   0.08697365 -0.166808
  0.10045415 -0.04047092  0.13048479  0.07793179]


In [4]:
center_word= "love"

center_id = np.array([word2idx[center_word]])

logits = model(center_id)
pred_id = tf.argmax(logits, axis =1).numpy()[0]

print("Predicted context:", idx2word[pred_id])

ValueError: Exception encountered when calling CBOW.call().

[1mInput 0 with name 'None' of layer 'dense_1' is incompatible with the layer: expected min_ndim=2, found ndim=1. Full shape received: (1,)[0m

Arguments received by CBOW.call():
  • x=tf.Tensor(shape=(1,), dtype=int64)

In [12]:
from sklearn.metrics.pairwise import cosine_similarity

word = "learning"
vec = embeddings[word2idx[word]].reshape(1,-1)

similarity = cosine_similarity(vec, embeddings)

for i in similarity.argsort()[0][-5:][::-1]:
  print(idx2word[i], similarity[0][i])


learning 1.0
machine 0.565871
and 0.40195066
love 0.1999953
i 0.08107284


Skip gram

In [13]:
import tensorflow as tf
import numpy as np

# Data
text = "i love deep learning and i love machine learning"
tokens = text.split()

vocab = sorted(set(tokens))
word2idx = {w:i for i, w in enumerate(vocab)}
idx2word = {i:w for w, i in word2idx.items()}

vocab_size = len(vocab)
window_size =2

# Skip gram data
def make_skipgram_data(tokens, window):
  X, y = [],[]

  for i in range(window_size, len(tokens)-window_size):
    center = word2idx[tokens[i]]

    for j in range(-window_size, window_size+1):
      if j !=0:
        context = word2idx[tokens[i+j]]
        X.append(center)
        y.append(context)

  return np.array(X), np.array(y)

X, y = make_skipgram_data(tokens, window_size)

# SKIPGRAM MODEL
embedding_dim = 10

class SkipGram(tf.keras.Model):
  def __init__(self):
    super().__init__()
    self.embedding = tf.keras.layers.Embedding(
        vocab_size, embedding_dim
        )
    self.output_layer = tf.keras.layers.Dense(vocab_size)

  def call(self, x):
    x = self.embedding(x)       # embed center word
    x = self.output_layer(x)    # predict context word
    return x

model = SkipGram()

model.compile(
    optimizer = "adam",
    loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics = ["accuracy"]
)

model.fit(X, y, epochs = 300, verbose = 0)

embeddings_skipgram = model.embedding.get_weights()[0]

In [14]:
center_word = "love"

center_id = np.array([word2idx[center_word]])

logits = model(center_id)
pred_id = tf.argmax(logits, axis =1).numpy()[0]

print("Predicted context:", idx2word[pred_id])

Predicted context: learning


In [15]:
# Word Vectors
embeddings = model.embedding.get_weights()[0]

print("Vector for 'love':")
print(embeddings[word2idx["love"]])

Vector for 'love':
[ 0.11656775  0.26377743  0.3802233   0.13177949  0.05273194 -0.1308903
 -0.09396358 -0.1903745   0.15693067 -0.10263672]


In [16]:
# Similarity
from sklearn.metrics.pairwise import cosine_similarity

word = "love"
vec = embeddings[word2idx[word]].reshape(1,-1)

similarity = cosine_similarity(vec, embeddings)

for i in similarity.argsort()[0][-5:][::-1]:
  print(idx2word[i], similarity[0][i])

love 0.99999994
i 0.35537183
deep 0.26409304
machine 0.24132475
learning -0.5650758
